# 阶段一实操：BOP 真值位姿下的 CAD 投影核验

## 阶段目标

BOP 标注提供了物体实例的真值位姿。本练习读取对应 CAD 模型和相机参数，将 CAD 几何从物体坐标系变换到相机坐标系，再投影到原始 RGB 图像。

| 已知输入 | 处理流程 | 结果 |
|---|---|---|
| RGB 图像、CAD 模型、相机内参 K、真值位姿 T_CO | CAD 表面采样 → 模型坐标 → 相机坐标 → 像素坐标 | 物体/相机坐标系中的 CAD 点云图，以及叠加包围盒和 XYZ 坐标轴的图像 |

若投影叠加与目标位置大致一致，可初步核对模型编号、位姿、相机参数和单位是否匹配。该几何链路也可用于后续检查算法输出。

**本阶段不训练网络、不估计位姿、不使用深度图。**练习内容是核对数据和几何计算链路。最后的回算误差仅用于检查公式实现的一致性，不能证明位姿算法精度或标注准确性。

本练习将坐标变换和点云基础应用于真实 BOP 样本。

## 数据和文件位置

Notebook 使用已准备好的 YCB-V BOP'19-24 子集。解压后的数据根目录位于项目目录：

~~~text
.local/data/bop/ycbv/
├── models/
│   ├── models_info.json
│   └── obj_000001.ply
└── test/
    └── <scene_id>/
        ├── rgb/
        ├── depth/
        ├── scene_camera.json
        ├── scene_gt.json
        └── scene_gt_info.json
~~~

`test` 下的数值文件夹表示场景；场景中的每张图像由 image id 标识，同一张图像可以包含多个物体实例。本阶段不读取深度图。该 test 子集仅用于学习和可视化，不用于训练模型。

## 准备：导入工具并设置数据路径

本单元导入所需库并设置数据集根目录。

In [ ]:
import json
from pathlib import Path

import numpy as np
import open3d as o3d
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "6D位姿估计":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_ROOT = PROJECT_ROOT / ".local" / "data" / "bop" / "ycbv"
SPLIT_DIR = DATASET_ROOT / "test"
MODELS_DIR = DATASET_ROOT / "models"

# None 表示扫描 test 下的全部场景；整数表示只扫描指定场景。
SCENE_ID = None


## 1. 选择场景、图像和实例

`scene_gt.json` 按 image id 保存实例标注，每张图像对应一个实例列表。默认扫描 `test` 下的全部场景，并选择 `visib_fract` 最高的实例。场景范围可通过准备单元中的 `SCENE_ID` 设置；`None` 表示扫描全部场景。

`gt_id` 是实例在当前图像标注列表中的序号；`obj_id` 是物体模型编号。代码会打印所选场景、图像、实例编号及其对应关系。

In [ ]:
def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


scene_dirs = sorted(
    path for path in SPLIT_DIR.iterdir()
    if path.is_dir() and path.name.isdigit()
)
if SCENE_ID is not None:
    scene_dirs = [path for path in scene_dirs if path.name == f"{SCENE_ID:06d}"]
if not scene_dirs:
    raise FileNotFoundError(f"在 {SPLIT_DIR} 中没有找到符合条件的场景目录。")

best = None
for candidate_scene_dir in scene_dirs:
    candidate_gt = read_json(candidate_scene_dir / "scene_gt.json")
    candidate_info = read_json(candidate_scene_dir / "scene_gt_info.json")

    for image_key, instances in candidate_gt.items():
        for candidate_gt_id, instance in enumerate(instances):
            info = candidate_info[image_key][candidate_gt_id]
            visibility = float(info.get("visib_fract", 0.0))
            if best is None or visibility > best[0]:
                best = (
                    visibility,
                    candidate_scene_dir,
                    int(image_key),
                    candidate_gt_id,
                    instance,
                    info,
                )

if best is None:
    raise ValueError("所选场景范围内没有可用的实例标注。")

visibility, scene_dir, image_id, gt_id, gt, gt_info = best
scene_gt = read_json(scene_dir / "scene_gt.json")
scene_camera = read_json(scene_dir / "scene_camera.json")
camera = scene_camera[str(image_id)]
obj_id = int(gt["obj_id"])

print(f"场景文件夹: {scene_dir.name}")
print(f"图像编号: {image_id:06d}")
print(f"选中实例序号 gt_id: {gt_id}")
print(f"模型编号 obj_id: {obj_id}")
print(f"该实例可见比例: {visibility:.3f}")
print("这张图像中的所有实例 (gt_id -> obj_id):")
for instance_id, instance in enumerate(scene_gt[str(image_id)]):
    marker = "  <-- 当前选中" if instance_id == gt_id else ""
    print(f"  {instance_id} -> {int(instance['obj_id'])}{marker}")

### 编号说明

**实例（instance）**指图像中一个具体物体的一次单独出现。标注列表中的每一条记录描述一个实例；一张图中有多个被标注的物体，就有多个实例。若同一个 CAD 模型对应的物体在画面中出现两次，那仍是两个实例。

`gt_id` 是实例在当前图像标注列表中的序号，从 0 开始；`obj_id` 是物体模型编号，用来选择 CAD 文件。例如 `gt_id=3 -> obj_id=10` 表示第 4 条记录描述了一个 `obj_id=10` 的实例，使用 `models/obj_000010.ply`。

同一个 `obj_id` 可以对应图中的多个实例：它们共用同一 CAD 模型，但每个实例有自己单独的 `gt_id` 和真值位姿。

## 2. 根据 obj_id 读取对应 CAD 模型

本节根据 `obj_id` 定位 `models/obj_XXXXXX.ply`，并读取模型尺寸信息。

In [ ]:
model_path = MODELS_DIR / f"obj_{obj_id:06d}.ply"
models_info = read_json(MODELS_DIR / "models_info.json")
model_info = models_info[str(obj_id)]

mesh = o3d.io.read_triangle_mesh(str(model_path))
vertices = np.asarray(mesh.vertices)

print(f"obj_id {obj_id} 对应的 CAD 文件: {model_path}")
print(f"顶点数: {len(vertices)}")
print(f"模型包围盒最小值 (mm): {vertices.min(axis=0)}")
print(f"模型包围盒最大值 (mm): {vertices.max(axis=0)}")
print(f"模型直径 (mm): {model_info['diameter']}")


## 3. 读取真值位姿和当前帧相机参数

场景标注中的 `cam_R_m2c` 和 `cam_t_m2c` 组成 $T_{CO}$，用于将物体坐标系 O 中的点变换到相机坐标系 C：

$$
\mathbf{p}_C = R_{CO}\mathbf{p}_O + \mathbf{t}_{CO}
$$

相机内参 `cam_K` 从同一图像的 `scene_camera.json` 读取。BOP 模型坐标和平移量的单位均为毫米。

In [ ]:
R_co = np.asarray(gt["cam_R_m2c"], dtype=float).reshape(3, 3)
t_co = np.asarray(gt["cam_t_m2c"], dtype=float).reshape(3)  # mm
K = np.asarray(camera["cam_K"], dtype=float).reshape(3, 3)

T_co = np.eye(4)
T_co[:3, :3] = R_co
T_co[:3, 3] = t_co

print("物体到相机的 T_CO:")
print(T_co)
print("相机内参 K:")
print(K)
print(f"平移 t_CO (mm): {t_co}")


## 4. CAD 点云在位姿变换前后的坐标系

CAD 网格由三角面片组成。本节从 CAD 表面均匀采样点。左图显示物体坐标系 O 中的点云；右图显示同一批点经真值 $T_{CO}$ 变换后，在相机坐标系 C 中的位置。

右图不是深度图恢复出的观测点云，也不是算法估计结果。它展示了已知真值位姿对 CAD 点的变换作用。后续单元将 CAD 包围盒投影到二维图像。

In [ ]:
# 设置从 CAD 表面抽取的点数。6000 是便于观察的采样数量，不是 CAD 网格原有的顶点数；
# 数量调大时点云会更密，但绘图和交互也会更慢。
CAD_POINT_COUNT = 6000

# 按三角网格表面面积均匀采样 CAD 表面，返回一个 Open3D 点云对象。
# 这里得到的是稀疏的表面点表示，尚未应用 T_CO，所以点仍处于物体坐标系 O。
cad_cloud = mesh.sample_points_uniformly(number_of_points=CAD_POINT_COUNT)

# 取出点云中的三维坐标并转为 NumPy 数组，形状为 (CAD_POINT_COUNT, 3)：
# 每一行是一个点，每列依次是该点在 O 系中的 X、Y、Z 坐标；本数据集模型单位为毫米。
cad_points_o = np.asarray(cad_cloud.points)

# 用 BOP 真值位姿，将同一批点从物体坐标系变到相机坐标系
cad_points_c = (R_co @ cad_points_o.T).T + t_co

# 为 3D 点云图准备物体坐标系中的 XYZ 轴
axis_len = 0.25 * model_info["diameter"]  # mm
axes_o = np.array([
    [0, 0, 0],
    [axis_len, 0, 0],
    [0, axis_len, 0],
    [0, 0, axis_len],
])
axes_c = (R_co @ axes_o.T).T + t_co

fig_cloud = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("CAD 点云：物体坐标系 O", "CAD 点云：相机坐标系 C"),
)

fig_cloud.add_trace(go.Scatter3d(
    x=cad_points_o[:, 0],
    y=cad_points_o[:, 1],
    z=cad_points_o[:, 2],
    mode="markers",
    marker=dict(size=2, color="royalblue"),
    name="CAD 点",
), row=1, col=1)

fig_cloud.add_trace(go.Scatter3d(
    x=cad_points_c[:, 0],
    y=cad_points_c[:, 1],
    z=cad_points_c[:, 2],
    mode="markers",
    marker=dict(size=2, color="royalblue"),
    name="变换后的 CAD 点",
), row=1, col=2)

axis_colors = ["red", "limegreen", "deepskyblue"]
axis_names = ["X", "Y", "Z"]
for axis_id, color, name in zip((1, 2, 3), axis_colors, axis_names):
    fig_cloud.add_trace(go.Scatter3d(
        x=[axes_o[0, 0], axes_o[axis_id, 0]],
        y=[axes_o[0, 1], axes_o[axis_id, 1]],
        z=[axes_o[0, 2], axes_o[axis_id, 2]],
        mode="lines",
        line=dict(color=color, width=5),
        name=f"{name}_O",
    ), row=1, col=1)
    fig_cloud.add_trace(go.Scatter3d(
        x=[axes_c[0, 0], axes_c[axis_id, 0]],
        y=[axes_c[0, 1], axes_c[axis_id, 1]],
        z=[axes_c[0, 2], axes_c[axis_id, 2]],
        mode="lines",
        line=dict(color=color, width=5),
        name=f"{name}_O in C",
    ), row=1, col=2)

fig_cloud.update_layout(
    title=f"obj_{obj_id:06d}: CAD 点云与真值位姿变换",
    height=650,
    width=1200,
    scene=dict(
        xaxis_title="X_O (mm)",
        yaxis_title="Y_O (mm)",
        zaxis_title="Z_O (mm)",
        aspectmode="data",
    ),
    scene2=dict(
        xaxis_title="X_C (mm)",
        yaxis_title="Y_C (mm)",
        zaxis_title="Z_C (mm)",
        aspectmode="data",
    ),
    margin=dict(l=10, r=10, t=60, b=10),
)
fig_cloud.show()


## 5. CAD 包围盒角点的像素投影

根据 CAD 包围盒参数构造 8 个角点。每个角点依次经过以下变换：

1. 使用真值 $T_{CO}$ 从物体坐标系变换到相机坐标系。
2. 使用相机内参 $K$ 投影到图像像素坐标。

透视投影需要除以相机深度 $Z$。若某角点的 $Z$ 小于或等于 0，表示该点位于相机后方，应检查坐标方向或位姿定义。

In [ ]:
x0, y0, z0 = (model_info[k] for k in ["min_x", "min_y", "min_z"])
x1 = x0 + model_info["size_x"]
y1 = y0 + model_info["size_y"]
z1 = z0 + model_info["size_z"]

bbox_o = np.array([
    [x0, y0, z0], [x1, y0, z0], [x1, y1, z0], [x0, y1, z0],
    [x0, y0, z1], [x1, y0, z1], [x1, y1, z1], [x0, y1, z1],
])


def project_model_points(points_o):
    points_c = (R_co @ points_o.T).T + t_co
    if np.any(points_c[:, 2] <= 0):
        raise ValueError("存在相机深度 Z <= 0 的点；检查位姿方向。")
    pixels_h = (K @ points_c.T).T
    pixels = pixels_h[:, :2] / pixels_h[:, 2:3]
    return points_c, pixels


bbox_c, bbox_uv = project_model_points(bbox_o)
print("角点相机深度 Z (mm):")
print(np.round(bbox_c[:, 2], 1))
print("角点像素位置 (u, v):")
print(np.round(bbox_uv, 1))


## 6. 在原始 RGB 图像中显示投影

黄色线框表示完整 CAD 包围盒的投影；红、绿、蓝线分别表示物体坐标系的 X、Y、Z 轴。物体被遮挡时，完整模型的投影范围可能大于图像中可见的部分。

图中显示的是使用真值位姿计算出的 CAD 投影，不是检测或估计算法的输出。

In [ ]:
rgb_path = scene_dir / "rgb" / f"{image_id:06d}.png"
if not rgb_path.exists():
    rgb_path = scene_dir / "rgb" / f"{image_id:06d}.jpg"

rgb = np.asarray(o3d.io.read_image(str(rgb_path)))[:, :, :3]
height, width = rgb.shape[:2]

bbox_edges = [
    (0, 1), (1, 2), (2, 3), (3, 0),
    (4, 5), (5, 6), (6, 7), (7, 4),
    (0, 4), (1, 5), (2, 6), (3, 7),
]

_, axes_uv = project_model_points(axes_o)

fig = go.Figure()
fig.add_trace(go.Image(z=rgb))

for i, j in bbox_edges:
    fig.add_trace(go.Scatter(
        x=[bbox_uv[i, 0], bbox_uv[j, 0]],
        y=[bbox_uv[i, 1], bbox_uv[j, 1]],
        mode="lines",
        line=dict(color="yellow", width=2),
        showlegend=False,
    ))

for axis_id, color in enumerate(["red", "lime", "deepskyblue"], start=1):
    fig.add_trace(go.Scatter(
        x=[axes_uv[0, 0], axes_uv[axis_id, 0]],
        y=[axes_uv[0, 1], axes_uv[axis_id, 1]],
        mode="lines+markers",
        line=dict(color=color, width=4),
        showlegend=False,
    ))

fig.update_xaxes(range=[0, width], title="u（像素）")
fig.update_yaxes(range=[height, 0], title="v（像素）", scaleanchor="x")
fig.update_layout(
    title=f"场景 {scene_dir.name}，图像 {image_id:06d}，模型 obj_{obj_id:06d}",
    width=900,
    height=700,
    margin=dict(l=20, r=20, t=50, b=20),
)
fig.show()


## 7. 投影结果的代数回算

像素坐标不包含深度，因此反投影时保留投影前的相机深度 $Z$。恢复到相机坐标后，再使用 $T_{CO}$ 的逆变换回到物体坐标。回算误差应接近浮点数精度。

该步骤检查投影、反投影和逆变换公式实现的一致性。由于计算往返使用相同的 $K$、$R$ 和 $t$，结果不能独立验证真值标注或位姿估计算法。

In [ ]:
pixels_h = np.c_[bbox_uv, np.ones(len(bbox_uv))]
rays_c = (np.linalg.inv(K) @ pixels_h.T).T
bbox_c_back = rays_c * bbox_c[:, 2:3]
bbox_o_back = (R_co.T @ (bbox_c_back - t_co).T).T

errors_mm = np.linalg.norm(bbox_o_back - bbox_o, axis=1)
print("8 个角点的回算误差 (mm):")
print(errors_mm)
print(f"最大回算误差 (mm): {errors_mm.max():.3e}")


## 本阶段小结

本阶段的数据和几何链路包括：

- 场景文件夹、图像编号与当前图像的实例列表；
- `gt_id`、`obj_id` 及 `obj_id` 到 `models/obj_XXXXXX.ply` 的对应关系；
- 真值位姿 $T_{CO}$ 将 CAD 点云从物体坐标系变换到相机坐标系；
- 左右两幅 CAD 点云图各自对应的坐标系；
- 相机参数 $K$ 将三维点投影到像素坐标；
- 真值几何链路核验与位姿估计精度之间的区别。

### 自测问题

1. 如果将 `gt_id` 当作模型编号，会产生什么结果？
2. 同一个 `obj_id` 在一帧中出现两次时，哪些信息相同，哪些信息各自独立？
3. 若包围盒超出画面或尺寸明显不符，应优先检查哪些项目？
4. 为什么回算误差很小仍不能作为位姿估计算法精度的证据？

完成自测后，可进入阶段二：使用深度图生成观测点云，并与 CAD 点云比较。